In [12]:
## 1. Load & check data
# Positive HAB events with Copernicus features. ~0.3% of rows have all ocean
# features missing (failed extraction or box on land).

import pandas as pd
import numpy as np

pos = pd.read_csv("extracted_positive_features.csv")
raw = pd.read_csv("harmful_algae_events.csv", low_memory=False)

df = pos.copy()
df["date"] = pd.to_datetime(df["date"])

print(df.shape)
df.isna().mean().sort_values(ascending=False).head(5)

(9638, 27)


,0
so_mean,0.003113
thetao_mean,0.003113
thetao_p95,0.003113
so_p95,0.003113
so_var,0.003113


In [13]:
## 2. Date precision check
# HAEDAT uses the 1st of the month when the exact day is unknown,
# and Jan 1 when only the year is known. Pandas still treats these as valid dates.
#
# Evidence:
# - ~640 events fall on day 1 vs. ~300 on other days
# - Northern Hemisphere has an unusual January spike (454 vs. 191 in December)
# - Raw records confirm many 1st-of-month dates have unknown exact days

hemi = np.where(df["Latitude"] >= 0, "N", "S")
display(df.groupby([hemi, df["date"].dt.month]).size().unstack(0).rename_axis(index="month", columns="hemi"))
display(df["date"].dt.day.value_counts().sort_index().head(5).rename_axis("day"))

hemi,N,S
month,,
1,454,477
2,246,348
3,389,387
4,422,374
5,541,421
6,643,289
7,753,294
8,591,308
9,443,357


,count
day,
1,640
2,315
3,262
4,288
5,326


In [14]:
## 3. Flag date precision
# Instead of dropping imprecise dates, we flag them based on the raw HAEDAT dates.
#
# day = exact date
# month = 1st of month, so exact day is unknown
# long_span = 60+ day span starting on the 1st, likely a placeholder
# ambiguous = Jan 1 with no end date, so the actual date is unclear
#
# Long blooms with a real start day stay as "day" since the start date is precise.
# We'll later compare models with and without the flagged dates.

span = (pd.to_datetime(raw["End Date"], errors="coerce")
        - pd.to_datetime(raw["Start Date"], errors="coerce")).dt.days

df = df.drop(columns="span_days", errors="ignore").merge(
    raw[["Event ID"]].assign(span_days=span), on="Event ID", how="left", validate="many_to_one")

first_of_month = df["date"].dt.day == 1
jan1 = first_of_month & (df["date"].dt.month == 1)

df["date_precision"] = "day"
df.loc[first_of_month, "date_precision"] = "month"
df.loc[jan1 & df["span_days"].isna(), "date_precision"] = "ambiguous"
df.loc[first_of_month & (df["span_days"] > 60), "date_precision"] = "long_span"
df["date_precision"].value_counts()

,count
date_precision,
day,8998
month,456
long_span,131
ambiguous,53


In [15]:
## 4. Drop rows with no ocean data
ocean_cols = [c for c in df.columns if c.endswith(("_mean", "_p95", "_var"))]

empty = df[ocean_cols].isna().all(axis=1)
print("empty rows dropped:", empty.sum())
df = df[~empty].copy()

empty rows dropped: 30


In [16]:
## 5. Check duplicates before removing them

# Some rows have the same location and date but different Event IDs. Before removing them, we want to see if they're duplicates or actually different events.

# What we found:
# - Raw HAEDAT has one row per Event ID, so our merges don't create extra rows
# - 1,848 rows are in 805 groups with the same location + date
# - Most are the same bloom reported more than once
# - Some have different species, toxin status, or start/end dates
# - Ocean features are identical within each group

keys = ["Latitude", "Longitude", "date"]
print("duplicate raw Event IDs:", raw["Event ID"].duplicated().sum())

dup_groups = df[df.duplicated(subset=keys, keep=False)].merge(
    raw[["Event ID", "Causative Species", "Toxin Detected?", "Start Date", "End Date"]],
    on="Event ID", how="left", validate="many_to_one")

g = dup_groups.groupby(keys)
summary = pd.DataFrame({
    "n_species": g["Causative Species"].nunique(),
    "n_toxin_status": g["Toxin Detected?"].nunique(),
    "n_start": g["Start Date"].nunique(),
    "n_end": g["End Date"].nunique(),
})
print("rows in duplicate groups:", len(dup_groups))
print("duplicate groups:", len(summary))
print("groups with >1 species:", (summary["n_species"] > 1).sum())
print("groups with mixed toxin status:", (summary["n_toxin_status"] > 1).sum())
print("groups with different start/end dates:", ((summary["n_start"] > 1) | (summary["n_end"] > 1)).sum())
print("max unique values per feature within a group:", g[ocean_cols].nunique().max().max())

duplicate raw Event IDs: 0
rows in duplicate groups: 1848
duplicate groups: 804
groups with >1 species: 70
groups with mixed toxin status: 34
groups with different start/end dates: 85
max unique values per feature within a group: 1


In [17]:
## 6. Collapse duplicates into one row per location + date
# We keep the original Event IDs, number of reports, toxin count, and species for analysis, but don't use them as model features.

df = df.merge(
    raw[["Event ID", "Toxin Detected?", "Causative Species"]],
    on="Event ID", how="left", validate="many_to_one"
)

merged_info = df.groupby(keys).agg(
    event_ids=("Event ID", lambda s: "; ".join(map(str, sorted(s.unique())))),
    n_records=("Event ID", "count"),
    n_toxic=("Toxin Detected?", lambda s: (s == "Yes").sum()),
    species=("Causative Species", lambda s: "; ".join(sorted(s.dropna().unique()))),
).reset_index()

df = (df.drop_duplicates(subset=keys, keep="first")
        .drop(columns=["Toxin Detected?", "Causative Species"])
        .merge(merged_info, on=keys, how="left", validate="one_to_one"))

print(df.shape)
df["date_precision"].value_counts()

(8564, 33)


,count
date_precision,
day,8010
month,399
long_span,112
ambiguous,43


In [18]:
## 7. Save the cleaned data
df.to_csv("positives_clean_v1.csv", index=False)

In [19]:
## 8. Build features

# We put the feature engineering in `build_features()` so the positive and negative data go through the same steps.

# - `doy_sin/cos` = time of year, treated as a cycle so Dec and Jan are close
# - `day_length` = daylight hours for the location and date
# - `current_speed` = how fast the water is moving
# - `temp_strat` = surface temp minus bottom temp
# - `temp_x_mld` = temperature × mixed layer depth

def day_length(lat, doy):
    decl = 23.44 * np.sin(np.radians(360 / 365 * (doy - 81)))
    x = -np.tan(np.radians(lat)) * np.tan(np.radians(decl))
    return 24 / np.pi * np.arccos(np.clip(x, -1, 1))

def build_features(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    doy = df["date"].dt.dayofyear

    df["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    df["day_length"] = day_length(df["Latitude"], doy)

    df["current_speed"] = np.sqrt(df["uo_mean"]**2 + df["vo_mean"]**2)
    df["temp_strat"] = df["thetao_mean"] - df["bottomT_mean"]
    df["temp_x_mld"] = df["thetao_mean"] * df["mlotst_mean"]

    return df

df_feat = build_features(df)
df_feat[["day_length", "current_speed", "temp_strat", "temp_x_mld"]].describe()

,day_length,current_speed,temp_strat,temp_x_mld
count,8564.000000,8564.000000,8564.000000,8564.000000
mean,12.770896,0.129611,14.055995,554.144506
std,2.059397,0.127090,10.197955,493.714803
min,5.040401,0.000601,-5.885489,-38.382272
25%,11.546873,0.061095,4.159167,206.720744
50%,12.358683,0.098871,13.082432,357.731527
75%,13.463686,0.155140,24.431658,745.177264
max,24.000000,1.246379,29.093933,3532.144085


In [20]:
## 9. Check for redundant features
feature_cols = [c for c in df_feat.columns if c.endswith(("_mean", "_p95", "_var"))] + \
               ["doy_sin", "doy_cos", "day_length", "current_speed", "temp_strat", "temp_x_mld"]

corr = df_feat[feature_cols].corr().abs()
pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
             .stack().sort_values(ascending=False))
pairs[pairs > 0.9]

thetao_mean  thetao_p95    0.997603
zos_mean     zos_p95       0.996651
so_mean      so_p95        0.983405
mlotst_mean  mlotst_p95    0.935113
             temp_x_mld    0.928662
dtype: float64

In [21]:
## 10. Save the features
df_feat.to_csv("positives_features_v1.csv", index=False)
print(df_feat.shape)

(8564, 39)


In [22]:
## 11. Load and check the negatives

neg_raw = pd.read_csv("extracted_negative_features.csv")   # original, never modified
neg = neg_raw.copy()
neg["date"] = pd.to_datetime(neg["date"])

print("shape:", neg.shape)
print("columns only in positives:", set(pos.columns) - set(neg.columns))
print("columns only in negatives:", set(neg.columns) - set(pos.columns))
print("date range:", neg["date"].min(), "to", neg["date"].max())
print("labels:", neg["label"].value_counts().to_dict())
neg.isna().mean().sort_values(ascending=False).head(5)

shape: (9618, 27)
columns only in positives: set()
columns only in negatives: set()
date range: 1993-01-03 00:00:00 to 2026-05-28 00:00:00
labels: {0: 9618}


,0
so_mean,0.003951
thetao_mean,0.003951
thetao_p95,0.003951
so_p95,0.003951
so_var,0.003951


In [25]:
## 11b. Check where the negatives came from
# Remove any negative that matches a positive location + date.

pos_sites = df[["Latitude", "Longitude"]].drop_duplicates()
at_pos_sites = neg.merge(pos_sites, on=["Latitude", "Longitude"]).shape[0]
print("negatives at a positive site:", at_pos_sites, "of", len(neg))

overlap = neg.merge(df[keys], on=keys).shape[0]
print("negatives matching a positive location + date:", overlap)

m = neg.merge(df[["Latitude", "Longitude", "date"]], on=["Latitude", "Longitude"], suffixes=("", "_pos"))
m["gap_days"] = (m["date"] - m["date_pos"]).dt.days.abs()
nearest = m.groupby(keys)["gap_days"].min()

print(nearest.describe())
print("within 7 days of a bloom:", (nearest <= 7).sum())
print("within 30 days of a bloom:", (nearest <= 30).sum())

negatives at a positive site: 9580 of 9618
negatives matching a positive location + date: 0
count     9580.000000
mean       371.049061
std       1092.591206
min         31.000000
25%         47.000000
50%         83.500000
75%        168.000000
max      11301.000000
Name: gap_days, dtype: float64
within 7 days of a bloom: 0
within 30 days of a bloom: 0


In [28]:
## 12. Clean the negatives
# Same steps as the positives: drop empty extractions and collapse duplicates.

neg = neg[~neg[ocean_cols].isna().all(axis=1)].copy()
neg = neg.drop_duplicates(subset=keys, keep="first")
print(neg.shape)

(9580, 27)


In [29]:
## 13. Combine positives and negatives

neg_feat = build_features(neg)
neg_feat["date_precision"] = "day"

full = pd.concat([df_feat, neg_feat], ignore_index=True)

print(full.shape)
print(full["label"].value_counts())
full.to_csv("modeling_dataset_v1.csv", index=False)

(18144, 39)
label
0    9580
1    8564
Name: count, dtype: int64
